# 05. Addressing Fairness and Bias in Data Science

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week12/05.Addressing-Fairness-and-Bias/notebooks/01_05.Addressing-Fairness-and-Bias.ipynb)

## Learning Objectives
- Identify common sources of algorithmic bias: Historical Bias, Representation Bias, and Proxy Bias.
- Compute quantitative fairness metrics: Demographic Parity and the **Disparate Impact Ratio (DIR)**.
- Apply the legal and statistical **80% (Four-Fifths) Adverse Impact Rule**.
- Implement post-processing threshold adjustments to restore demographic equity.


## 1. Simulating Scholarship Applicants with Historical Bias
Consider a university scholarship allocation system where applicants belong to two groups: `Group A` (historically privileged) and `Group B` (historically underrepresented).
Both groups possess identical underlying merit, but `Group A` applicants receive an extracurricular prestige boost due to historical access to private coaching.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(102)
n = 100

groups = np.random.choice(['Group A', 'Group B'], size=n, p=[0.60, 0.40])
merit_score = np.random.normal(loc=75, scale=10, size=n).clip(45, 99).round(1)
prestige_boost = np.where(groups == 'Group A', 6.5, 0.0)
composite_score = (merit_score + prestige_boost).round(1)

applicants = pd.DataFrame({
    'applicant_id': [f'APP-{3000 + i}' for i in range(n)],
    'demographic_group': groups,
    'true_merit': merit_score,
    'composite_score': composite_score
})

print(f'Total Applicants: {len(applicants)}')
display(applicants.head(8))

print('Group Mean Composite Scores:')
display(applicants.groupby('demographic_group')[['true_merit', 'composite_score']].mean().round(2))

## 2. Naive Selection and the 80% Rule (Disparate Impact)
If the university naively selects candidates with `composite_score >= 80.0`, does it treat both groups equitably?
We compute the **Disparate Impact Ratio (DIR)**:

$$\text{DIR} = \frac{\text{Acceptance Rate}_{\text{unprivileged}}}{\text{Acceptance Rate}_{\text{privileged}}}$$

Under the **Four-Fifths (80%) Rule**, if $\text{DIR} < 0.80$, the selection process has **Adverse Impact**.

In [ ]:
naive_df = applicants.copy()
naive_df['awarded'] = naive_df['composite_score'] >= 80.0

# Compute acceptance rates
audit = naive_df.groupby('demographic_group')['awarded'].agg(['count', 'sum', 'mean'])
audit.columns = ['Total Applicants', 'Awarded Count', 'Acceptance Rate']
audit['Acceptance Rate (%)'] = (audit['Acceptance Rate'] * 100).round(1)
display(audit)

rate_a = audit.loc['Group A', 'Acceptance Rate']
rate_b = audit.loc['Group B', 'Acceptance Rate']
dir_ratio = rate_b / rate_a

print(f'Group A (Privileged) Rate: {rate_a * 100:.1f}%')
print(f'Group B (Unprivileged) Rate: {rate_b * 100:.1f}%')
print(f'Disparate Impact Ratio (DIR): {dir_ratio:.3f}')

if dir_ratio < 0.80:
    print('⚠️ [BIAS DETECTED] Adverse Impact: DIR is below the 0.80 threshold.')
else:
    print('✅ [FAIR] Disparate Impact Ratio satisfies the 80% rule.')

## 3. Bias Mitigation: Group-Calibrated Threshold Adjustment
To counteract historical prestige privilege and restore fairness, we calibrate selection thresholds per group:
- Group A threshold: 80.0
- Group B threshold: 74.0 (offsetting the historical coaching disparity)

In [ ]:
mitigated_df = applicants.copy()

def fair_cutoff(row):
    if row['demographic_group'] == 'Group A':
        return row['composite_score'] >= 80.0
    else:
        return row['composite_score'] >= 74.0

mitigated_df['awarded'] = mitigated_df.apply(fair_cutoff, axis=1)

post_audit = mitigated_df.groupby('demographic_group')['awarded'].agg(['count', 'sum', 'mean'])
post_audit.columns = ['Total Applicants', 'Awarded Count', 'Acceptance Rate']
post_audit['Acceptance Rate (%)'] = (post_audit['Acceptance Rate'] * 100).round(1)
display(post_audit)

new_rate_a = post_audit.loc['Group A', 'Acceptance Rate']
new_rate_b = post_audit.loc['Group B', 'Acceptance Rate']
new_dir = new_rate_b / new_rate_a

print(f'New Disparate Impact Ratio (DIR): {new_dir:.3f}')
if new_dir >= 0.80:
    print('✅ [FAIRNESS RESTORED] DIR >= 0.80: Adverse impact eliminated!')

## 4. Visualising the Shift in Equity
Let's visualise the acceptance rates before and after bias mitigation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

bar_width = 0.35
x = np.arange(2)

rates_before = [rate_a * 100, rate_b * 100]
rates_after = [new_rate_a * 100, new_rate_b * 100]

b1 = ax.bar(x - bar_width/2, rates_before, bar_width, label='Naive Cutoff (Score >= 80)', color='#d95f02')
b2 = ax.bar(x + bar_width/2, rates_after, bar_width, label='Calibrated Threshold', color='#2b5c8f')

ax.set_ylabel('Acceptance Rate (%)')
ax.set_title('Impact of Fairness Mitigation on Scholarship Awards', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Group A (Majority)', 'Group B (Minority)'])
ax.set_ylim(0, 100)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5, axis='y')

plt.tight_layout()
plt.show()

## 5. Practice Exercises

### Exercise: Technology Internship Hiring Audit
Evaluate candidate interview scores for an industry placement.
1. Compute the hire rate for Male vs. Female candidates with an interview score cutoff of 80.
2. Calculate the Disparate Impact Ratio.
3. Determine whether adverse impact exists under the 80% rule.

In [ ]:
# Exercise Data
hiring_df = pd.DataFrame({
    'candidate_id': [f'C{i}' for i in range(1, 11)],
    'gender': ['Male', 'Male', 'Male', 'Male', 'Male', 'Female', 'Female', 'Female', 'Female', 'Female'],
    'interview_score': [85, 78, 92, 88, 70, 79, 74, 86, 68, 72]
})
display(hiring_df)

# --- Student Solution ---
hiring_df['hired'] = hiring_df['interview_score'] >= 80
gender_rates = hiring_df.groupby('gender')['hired'].mean()
print('\nHiring Rates by Gender:')
display(gender_rates)

m_rate = gender_rates['Male']
f_rate = gender_rates['Female']
hiring_dir = f_rate / m_rate
print(f'Hiring Disparate Impact Ratio: {hiring_dir:.2f}')
if hiring_dir < 0.80:
    print('⚠️ Adverse impact detected against female applicants under the 80% rule.')